# Soil Sample Spatial Interpolation - Ordinary/Universal Kriging
**Yuxi Zhang** · Oct 18, 2026 · [@yuxi-research](https://github.com/yuxi-research)


📘 This notebook spatially interpolates soil moisture and covariates using Universal Kriging with drift terms `DEM` and `Slope`.  It processes all sampled variables (SMVol, BD, Clay, C) across all dates and individual depth intervals, producing corresponding raster maps.  


⚙️ **Main Processing Steps (Ordinary/Universal Kriging)**  


1. **Read Farm Boundary**  
   - Load the cleaned `{farm_name}.shp` shapefile and convert it to the target coordinate reference system and projection (`epsg='{proj_crs}'`).


2. **Process DEM and Slope**  
   - Load clipped DEM and slope files for the specific farm, and convert them to the target CRS.  
   - Use the DEM file as the reference raster and reproject all other rasters (e.g., slope) to align with its grid.  
   - Save the reprojected rasters to `data_lv3/{farm_name}/kriged_ok_*/`.  
   - Visualize all projected rasters using `plot_raster`.  
   - *(Optional)*: Create a custom user-defined grid and affine transform, then reproject all rasters to this new grid.

3. **Run Universal Kriging**  
   - Read the soil sample data, selecting the variable, date, and depth to interpolate (e.g., SMVol 0–30 cm on 2021-04-28).  
   - Drop rows with NaN values.  
   - Extract point values from drift term rasters at soil sample locations (1D arrays: `dem_values`, `slope_values`).  
   - Construct Universal Kriging with 'specified' drift terms:  
     ```python
     drift_terms = ["specified", "specified"]
     specified_drift = [dem_values, slope_values]
     ```
   - Execute Universal Kriging with the specified 2D drift grids:  
     ```python
     z_grid, ss_grid = uk.execute(
         'grid', xgrid, ygrid, 
         specified_drift_arrays=[dem_proj.data, slope_proj.data]
     )
     ```
   - Convert the output to an XArray and save it as a GeoTIFF. *Note: the output is not masked by default.*  
   - 
   - Mask values outside the farm boundary and visualize the result. Include the farm boundary, sample points, and a histogram of the masked data.

💬 **Notes**
- The histogram may overlap the map in some cases. Adjust its position if necessary.

📥 **Input**  

- `./data_lv2/{farm_name}/data/`  
   - `{farm_name}_soilsamples_weighted.csv`: Sampled soil moisture and attributes of all dates, samples vertically weighted.
   - `{farm_name}_boundary_EPGS{proj_crs}.shp`: Projected farm boundary. 

📤 **Output**  

- `./data_lv2/{farm_name}/kriged_ok_weighted/`: Kriged maps of soil moisture and attributes, samples vetically weighted.

### Settings

In [1]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
import rioxarray as rxr
import xarray as xr
from rasterio.enums import Resampling
from pykrige.uk import UniversalKriging

display = False
save_pdf = True


In [2]:
# Path settings
from global_settings import *
farm_name = os.environ.get("FARM_NAME", "Amaroo")
dirs = ProjectPaths(farm_name=farm_name)
proj_crs = dirs.proj_crs
inset_loc = dirs.inset_loc
buffer_dis = BUFFER_DIS


#### Pipeline parameters

In [3]:
mode = 'ok'
depth_suffix = '_weighted'
dir_out = dirs.lv2_ok
dir_out_f = dir_out / 'figures'

os.makedirs(str(dir_out_f), exist_ok=True)


### Read data

In [4]:
# Read the soil moisture data
pth_soil = os.path.join(dirs.lv2_data, f'{farm_name}_soilsamples{depth_suffix}.csv')
df_soil = pd.read_csv(pth_soil)

geometry = gpd.points_from_xy(df_soil['Longitude'], df_soil['Latitude'], crs='EPSG:4326').to_crs(proj_crs)
gdf_soil = gpd.GeoDataFrame(df_soil, geometry=geometry, crs=proj_crs)
# gdf_soil[~gdf_soil['SMGrav'].isna()]

In [5]:
# Open the target grid raster to align the kriging results
grid_target = rxr.open_rasterio(os.path.join(dirs.lv2_data, f'target_grid.tif'),
    mask_and_scale=True).squeeze()
grid_target


## Ordinary/Universal kriging
Note: By disabling drift terms, the universal kriging reduces to ordinary kriging.

In [ ]:
# import rioxarray as rxr
# import xarray as xr
# import numpy as np
# from rasterio.enums import Resampling
# from pykrige.uk import UniversalKriging
# import pandas as pd
# from affine import Affine

# Read boundary
bound = gpd.read_file(os.path.join(dirs.lv1_shp, f'{farm_name}_boundary_epsg{proj_crs}.shp'))

######## Uncomment Below for Universal Kriging ########
# 1. Read DEM and slope
try:
    dem = rxr.open_rasterio(os.path.join(dirs.lv1_cov, 'DEM_clip.tif'), mask_and_scale=True).squeeze()
    slope = rxr.open_rasterio(os.path.join(dirs.lv1_cov, 'Slope_clip.tif'), mask_and_scale=True).squeeze()
    twi = rxr.open_rasterio(os.path.join(dirs.lv1_cov, 'TWI_clip.tif'), mask_and_scale=True).squeeze()

    # 2. Reproject to match the target grid. 
    dem_proj = dem.rio.reproject_match(grid_target, resampling=Resampling.bilinear)
    slope_proj = slope.rio.reproject_match(grid_target, resampling=Resampling.bilinear)
    twi_proj = twi.rio.reproject_match(grid_target, resampling=Resampling.bilinear)

except Exception as e:
    print(f"Error reading or reprojecting DEM, slope, or TWI: {e}")
    dem_exists = False

######## Uncomment Above for Universal Kriging ########

# 3. Extract resolution and extent info from the target grid
nx = grid_target.rio.width
ny = grid_target.rio.height
xgrid = grid_target.x.values
ygrid = grid_target.y.values
# A standard GIS coordinate system has the origin at the top-left corner, 
# with the x-axis pointing to the right and the y-axis pointing downwards.
# Here, y[0] represents the maximum value, and dy is a negative value.
# Note not to mix xgrid.min() with xgrid[1]-xgrid[0]
xmin = xgrid[0]; xmax = xgrid[-1]; dx = xgrid[1] - xgrid[0]
ymin = ygrid[-1]; ymax = ygrid[0]; dy = ygrid[1] - ygrid[0]


In [ ]:
# Save projected DEM and slope
try:
    dem_proj.rio.to_raster(os.path.join(dir_out, f'{farm_name}_static_DEM_proj.tif'), driver='GTiff', dtype='float32', nodata=np.nan)
    slope_proj.rio.to_raster(os.path.join(dir_out, f'{farm_name}_static_Slope_proj.tif'), driver='GTiff', dtype='float32', nodata=np.nan)
    twi_proj.rio.to_raster(os.path.join(dir_out, f'{farm_name}_static_TWI_proj.tif'), driver='GTiff', dtype='float32', nodata=np.nan)
    # Project NDVI and save
    # ndvi_proj = ndvi.rio.reproject_match(grid_target, resampling=Resampling.bilinear)
    # ndvi_proj.rio.to_raster(os.path.join(pth_out2_krig_ok, f'{farm_name}_{date}_NDVI_proj.tif'), driver='GTiff', dtype='float32', nodata=np.nan)
except Exception as e:
    print(f"Error saving projected DEM, slope, or TWI: {e}")

In [ ]:
def reverse_inset_loc(loc):
    if loc == 'upper':
        return 'bottom'
    elif loc == 'lower':
        return 'top'
    elif loc == 'left':
        return 'right'
    elif loc == 'right':
        return 'left' 


inset_loc_xaxis = reverse_inset_loc(inset_loc.split()[0])
inset_loc_yaxis = reverse_inset_loc(inset_loc.split()[1])

In [ ]:
dict_unit = {
    'DEM': 'm',
    'Slope': '°',
    'TWI': '',
    'C': r'$\%\ g\ g^{-1}$',
    'H': r'$\%\ g\ g^{-1}$',
    'BD': r'$g\ cm^{-3}$',
    'Clay': r'$\%\ g\ g^{-1}$',
    'SMGrav': r'$g\ g^{-1}$',
    'SMVol': r'$cm^3\ cm^{-3}$',
    'watLatticeVol': r'$cm^3\ cm^{-3}$',
    'watCarbonVol': r'$cm^3\ cm^{-3}$',
    'watHydrogenVol': r'$cm^3\ cm^{-3}$',
    'watLatticeGrav': r'$g\ g^{-1}$',
    'watCarbonGrav': r'$g\ g^{-1}$',
    'watHydrogenGrav': r'$g\ g^{-1}$',

    }

dict_unit.get("Clay", None) # Test


### Plot rasters

In [ ]:
# Function: plot_raster. Plot raster data with inset histogram.
def plot_raster(rxrdata, bound, varname=None, date_str='', hist=True, mask=True, cmap=custom_cmap, vmin=None, vmax=None):
    """
    Plot a raster with specified colormap and value range.
    """

    # Create a mask for the area outside the farm boundary
    arr = rxrdata.data.ravel()
    arr = arr[~np.isnan(arr)]
    vmin = arr.min() if vmin is None else vmin
    vmax = arr.max() if vmax is None else vmax
    unit = dict_unit.get(varname, None)  # Get unit from the dictionary, default to empty string if not found
    unit = f'({unit})' if unit else ''

    ### Mask the area outside the farm boundary with a buffer
    if mask:
        from rasterio.features import geometry_mask
        mask_geom = geometry_mask(bound.geometry.buffer(buffer_dis), 
                                # y increment is negative because the origin is at the top-left corner of the raster.
                                # Here ymax should be passed, and dy should be negative.
                                transform = rxrdata.rio.transform(),
                                                        all_touched=True, 
                                                        out_shape=rxrdata.shape)

        rxrdata = rxrdata.where(~mask_geom)

    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(8, 9), sharex='all', sharey='all')

    rxrdata.plot(ax=ax, cmap=cmap, add_colorbar=True, cbar_kwargs={'shrink': 0.6, 'label':f'{varname} {unit}'},
                 vmin=vmin, vmax=vmax)


    #### Axes settings
    bound.plot(ax=ax, facecolor='none', edgecolor='black', lw=1)  # Plot boundary
    ax.set_xlim(bound.total_bounds[0]-buffer_dis, bound.total_bounds[2]+buffer_dis)
    ax.set_ylim(bound.total_bounds[1]-buffer_dis, bound.total_bounds[3]+buffer_dis)
    ax.tick_params(labelsize=8, direction='in', length=4)  # Set tick parameters
    ax.set_xlabel('Easting (m)', fontsize=10)  # X label
    ax.set_ylabel('Northing (m)', fontsize=10)  # Y label
    # ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    ax.set_aspect('equal', adjustable='box')  # Set aspect ratio to equal

    ax.set_title(f'{varname} {farm_name} {date_str}' if varname else ax.get_title())
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    # plt.colorbar(label='Value')


    #### Add histogram to the upper left corner
    if True:
        from mpl_toolkits.axes_grid1.inset_locator import inset_axes
        color = 'deeppink'
        inset_ax = inset_axes(ax, width="30%", height="20%", borderpad=0, loc=inset_loc)
        # Plotting
        inset_ax.hist(arr, bins=20, color=color, alpha=1, edgecolor='white', lw=0.2, density=True)

        inset_ax.set_xlim(vmin, vmax)  # Set x-axis limits to be the same as the main plot
        # Set tick parameters
        inset_ax.tick_params(labelsize=6, direction='in', length=2, colors=color)  # Set tick parameters
        # Adjust the position of the y-axis label to right
        inset_ax.yaxis.set_ticks_position(inset_loc_yaxis)  # Y ticks on right
        inset_ax.yaxis.set_label_position(inset_loc_yaxis)  # Y label on to right
        inset_ax.xaxis.set_ticks_position(inset_loc_xaxis)  # Y ticks on right
        inset_ax.xaxis.set_label_position(inset_loc_xaxis)  # Y label on to right
        # Set spine colors
        inset_ax.spines[inset_loc_xaxis].set_color(color)
        inset_ax.spines[inset_loc_yaxis].set_color(color)
        # Set labels and title
        inset_ax.set_xlabel(f'{varname} ({unit})', color=color, fontsize=6, labelpad=1)  # X label
        # inset_ax.set_ylabel("Frequency", color=color, fontsize=6, labelpad=1)  # Y label
        inset_ax.set_ylabel("")  # Y label
        inset_ax.set_yticks([])  # Remove y ticks
        # inset_ax.set_title('Histogram', fontsize=6, pad=2, color=color)
        inset_ax.grid(True, which='both', linestyle='', linewidth=0.1)
        # inset_ax.legend(fontsize=6, loc=2)

    ### Save and show
    outfig = os.path.join(dir_out, f'{farm_name}_{date_str}_{varname}_proj.png')
    plt.savefig(outfig, bbox_inches='tight')
    if save_pdf:
        plt.savefig(os.path.join(dir_out_f, f'{farm_name}_{date_str}_{varname}_proj.pdf'))
    if display:
        plt.show()
    else:
        print('Figure saved to:', outfig)
        plt.close()

In [ ]:
# Plot the projected DEM, slope, and TWI
try:
    plot_raster(dem_proj, bound, 'DEM', 'static', cmap='terrain')
    plot_raster(slope_proj, bound, 'Slope', 'static', cmap='terrain')
    plot_raster(twi_proj, bound, 'TWI', 'static', cmap='Spectral')
    # plot_raster(ndvi_proj, bound, 'NDVI', date_str, cmap='RdYlGn') # RdYlGn, BrBG
except Exception as e:
    print(f"Error plotting raster: {e}")

In [ ]:
# gdf_soil.drop_duplicates().dropna(subset=['C', 'geometry'])

### Function to run Ordinary/Universal Kriging.
* The default method is Ordinary Kriging.
* To run Universal Kriging, uncomenent code blocks indicated below.

In [ ]:
# Function: run_uk. Run Universal Kriging.
# By disabling drift terms, the universal kriging reduces to ordinary kriging.
def run_uk(gdf, bound, varname, date_str, vmodel, hist=True, cmap=custom_cmap, vmin=None, vmax=None, suffix=''):
    outname = f'{farm_name}_{date_str}_{varname}{suffix}'
    # Drop duplicates and NaN values.
    if varname not in gdf.columns:
        print(f"Variable {varname} not found in the data. Skipping Universal Kriging.")
        return None

    gdf_dropna = gdf.drop_duplicates().dropna(subset=[varname, 'geometry'])
    if gdf_dropna.empty:
        print(f"No valid data points for {varname} on {date_str}. Skipping Universal Kriging.")
        return None

    ######## Uncomment Below for Universal Kriging - Part 1/3 ########

    # # Extract DEM and slope values at sample locations.
    # dem_values = extract_raster_values(gdf_dropna.geometry, dem_proj)
    # slope_values = extract_raster_values(gdf_dropna.geometry, slope_proj)

    ######## Uncomment Above for Universal Kriging - Part 1/3  ########

    # 5. Read coordinates and values for kriging
    x = gdf_dropna.geometry.x.values
    y =gdf_dropna.geometry.y.values
    z = gdf_dropna[varname].values
    print(f"{varname} XYZ shapes:", x.shape, y.shape, z.shape)
    
    if len(z) == 0:
        print(f"No data for {varname} on {date_str}. Skipping Universal Kriging.")
        return None
    
    if z.max() == z.min():
        print(f"All values for {varname} on {date_str} are the same. Skipping Universal Kriging.")
        return None

    # 6. Construct and execute Universal Kriging or Ordinary Kriging (by disabling drift terms)
    uk = UniversalKriging(
        x, y, z,
        variogram_model = vmodel,
        ######## Uncomment Below for Universal Kriging - Part 2/3 ########
        # drift_terms = ["specified", "specified"],  # For universal kriging with external drift
        # specified_drift = [dem_values, slope_values], # Pass the covariate values at sample locations
        ######## Uncomment Below for Universal Kriging - Part 2/3 ########

        # coordinates_type="euclidean",  # geographic
        nlags=10,  # Number of bins for the variogram
        exact_values=True,
        weight = True,
        verbose = display,
        enable_plotting = display,
    )
    
    # Note that the grid specified here is the same as the target grid for covariates, 
    # which ensures that the kriged output will be on the same grid as the covariates. 
    # This allows for direct comparison and combination of the kriged results with the covariate data.
    z_grid, ss_grid = uk.execute('grid', xgrid, ygrid, 
    ######## Uncomment Below for Universal Kriging - Part 3/3 ########
                                 #specified_drift_arrays=[dem_proj.data, slope_proj.data] # Pass the covariate 2D at target locations for universal kriging
    ######## Uncomment Below for Universal Kriging - Part 3/3 ########
                                 )

    # 7. Create a DataArray for the kriged grid and set the CRS
    z_da = xr.DataArray(
        z_grid,
        coords={"y": ygrid, "x": xgrid},
        dims=( "y", "x"),
        name=varname
    ).rio.write_crs(grid_target.rio.crs)

    # 8. Mask the area outside the farm boundary with a buffer.
    from rasterio.features import geometry_mask
    # from rasterio.transform import from_origin.
    mask_geom = geometry_mask(bound.geometry.buffer(buffer_dis), 
                            # y increment is negative because the origin is at the top-left corner of the raster.
                            # Here ymax should be passed, and dy should be negative.
                            transform = z_da.rio.transform(),   # Method 1: Directly use the existing transform
                            # transform = from_origin(xmin, ymax, dx, abs(dy)),     # Method 2: If using from_origin, note that dy should be positive. from_origin will convert it to negative automatically.
                                                    all_touched=True, 
                                                    out_shape=z_da.shape)
    z_da = z_da.where(~mask_geom)

    # 9. Export the DataArray to GeoTIFF
    outfilepth = os.path.join(dir_out, f"{outname}.tif")
    z_da.rio.to_raster(outfilepth)
    print(f'File saved to: {outfilepth}')


    ####### Plotting

    # Settings
    arr = z_da.data.ravel()
    arr = arr[~np.isnan(arr)]
    vmin = arr.min() if vmin is None else vmin
    vmax = arr.max() if vmax is None else vmax
    unit = dict_unit.get(varname, None)  # Get unit from the dictionary, default to empty string if not found
    unit = f'({unit})' if unit else ''

    # Plot map
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(8, 9), sharex='all', sharey='all')

    z_da.plot(ax=ax, cmap=cmap, add_colorbar=True, cbar_kwargs={'shrink': 0.6, 'label':f'{varname} {unit}'},
                 vmin=vmin, vmax=vmax)

    # Add soil sampling points
    gdf_dropna.plot(ax=ax, column=f'{varname}', marker='o', markersize=50, edgecolor='black', zorder=20,
                        cmap=cmap, legend=False, # alpha=0.3,
                        legend_kwds={'label': f'{varname}_sample {unit}', 'shrink': 0.6}, 
                        vmin=vmin, vmax=vmax
                        )

    #### Axes settings
    bound.plot(ax=ax, facecolor='none', edgecolor='black', lw=1)  # Plot boundary
    ax.set_xlim(bound.total_bounds[0]-buffer_dis, bound.total_bounds[2]+buffer_dis)
    ax.set_ylim(bound.total_bounds[1]-buffer_dis, bound.total_bounds[3]+buffer_dis)
    ax.tick_params(labelsize=8, direction='in', length=4)  # Set tick parameters
    ax.set_xlabel('Easting (m)', fontsize=10)  # X label
    ax.set_ylabel('Northing (m)', fontsize=10)  # Y label
    # ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    ax.set_aspect('equal', adjustable='box')  # Set aspect ratio to equal

    ax.set_title(f'{varname} {farm_name} {date_str}' if varname else ax.get_title())
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    # plt.colorbar(label='Value')


    #### Add histogram to corner
    if True:
        from mpl_toolkits.axes_grid1.inset_locator import inset_axes
        color = 'deeppink'
        inset_ax = inset_axes(ax, width="30%", height="20%", borderpad=0, loc=inset_loc)
        # Plotting
        inset_ax.hist(arr, bins=20, color=color, alpha=1, edgecolor='white', lw=0.2, density=True)

        inset_ax.set_xlim(vmin, vmax)  # Set x-axis limits to be the same as the main plot
        # Set tick parameters
        inset_ax.tick_params(labelsize=6, direction='in', length=2, colors=color)  # Set tick parameters
        # Adjust the position of the y-axis label to right
        inset_ax.yaxis.set_ticks_position(inset_loc_yaxis)  # Y ticks on right
        inset_ax.yaxis.set_label_position(inset_loc_yaxis)  # Y label on to right
        inset_ax.xaxis.set_ticks_position(inset_loc_xaxis)  # Y ticks on right
        inset_ax.xaxis.set_label_position(inset_loc_xaxis)  # Y label on to right
        # Set spine colors
        inset_ax.spines[inset_loc_xaxis].set_color(color)
        inset_ax.spines[inset_loc_yaxis].set_color(color)
        # Set labels and title
        inset_ax.set_xlabel(f'{varname} {unit}', color=color, fontsize=6, labelpad=1)  # X label
        # inset_ax.set_ylabel("Frequency", color=color, fontsize=6, labelpad=1)  # Y label
        inset_ax.set_ylabel("")  # Y label
        inset_ax.set_yticks([])  # Remove y ticks
        # inset_ax.set_title('Histogram', fontsize=6, pad=2, color=color)
        inset_ax.grid(True, which='both', linestyle='', linewidth=0.1)
        # inset_ax.legend(fontsize=6, loc=2)

    ### Save and show
    outfig = os.path.join(dir_out_f, f'{outname}.png')
    plt.savefig(outfig, bbox_inches='tight')
    if save_pdf:
        plt.savefig(os.path.join(dir_out_f, f'{outname}.pdf'))
    if display:
        plt.show()
    else:
        print('Figure saved to:', outfig)
        plt.close()

    return outfilepth  # Return the path of the saved GeoTIFF file

### Loop to run kriging for each date at bulk depth (vertical weighted average) 

In [ ]:
# Run bulk profile on weighted average soil moisture and other variables, without depth separation
outfiles = []
for date in gdf_soil['Date'].unique()[:]:
    gdf_soil_day = gdf_soil[gdf_soil['Date']==date].sort_values(['SiteID'])
    filename = f'{farm_name}_{date}'
    date_str = filename.split('_')[-1]
    print(filename)

    outfiles.append(run_uk(gdf_soil_day, bound, 'watHydrogenGrav', date_str, 'spherical', suffix=f'{depth_suffix}_{mode}'))
    outfiles.append(run_uk(gdf_soil_day, bound, 'watLatticeGrav', date_str, 'spherical', suffix=f'{depth_suffix}_{mode}'))
    outfiles.append(run_uk(gdf_soil_day, bound, 'watCarbonGrav', date_str, 'spherical', suffix=f'{depth_suffix}_{mode}'))
    outfiles.append(run_uk(gdf_soil_day, bound, 'SMGrav', date_str, 'spherical', suffix=f'{depth_suffix}_{mode}'))

    if 'SMVol' in gdf_soil_day.columns:
        outfiles.append(run_uk(gdf_soil_day, bound, 'watHydrogenVol', date_str, 'spherical', suffix=f'{depth_suffix}_{mode}'))
        outfiles.append(run_uk(gdf_soil_day, bound, 'watLatticeVol', date_str, 'spherical', suffix=f'{depth_suffix}_{mode}'))
        outfiles.append(run_uk(gdf_soil_day, bound, 'watCarbonVol', date_str, 'spherical', suffix=f'{depth_suffix}_{mode}'))
        outfiles.append(run_uk(gdf_soil_day, bound, 'SMVol', date_str, 'spherical', suffix=f'{depth_suffix}_{mode}'))

        print('SMVol not found, used SMGrav for soil moisture kriging.')

    # break

outfiles = [ofile for ofile in outfiles if ofile is not None]  # Filter out None values

## Check

In [ ]:
import datetime as dt
print('Last processed time:', dt.datetime.now().astimezone().isoformat())

In [ ]:
import utils.check_data as check
for outfile in outfiles:
    check.check_raster(outfile, show_info=False)